# Trustworthy Personalised AI — Analysis Dashboard

Research notebook for analysing training data quality, model benchmark results, and conversation samples. All charts are saved to `exports/` as SVG (vector, dissertation-ready) and PNG (high-resolution raster) via plotly + kaleido. Run cells top-to-bottom on first use; individual sections can be re-run independently after that.

In [ ]:
import json
import re
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from IPython.display import HTML, display

pio.templates.default = "plotly_white"
PALETTE = px.colors.qualitative.Set2

DATA_DIR    = Path("data")
REPORTS_DIR = Path("reports")
EXPORTS_DIR = Path("exports")
EXPORTS_DIR.mkdir(exist_ok=True)

# ---------------------------------------------------------------------------
# Dissertation style — A4 text width at 300 dpi
# A4 text width ≈ 160 mm; scale=3 PNG → 2250 px ≈ 281 dpi
# ---------------------------------------------------------------------------
A4_W = 750   # base figure width in px

_FONT      = dict(family="Arial, sans-serif", size=12, color="#1e293b")
_FONT_AXIS = dict(family="Arial, sans-serif", size=12, color="#334155")
_FONT_TICK = dict(family="Arial, sans-serif", size=10, color="#475569")
_FONT_TITL = dict(family="Arial, sans-serif", size=14, color="#0f172a")

_AXIS_STYLE = dict(
    showgrid=True,  gridcolor="#f1f5f9", gridwidth=1,
    showline=True,  linecolor="#cbd5e1", linewidth=1,
    tickfont=_FONT_TICK,
    title_font=_FONT_AXIS,
    zeroline=False,
)

def style_fig(fig, width=A4_W, height=420):
    """Apply dissertation-ready layout: A4 width, consistent fonts and grid."""
    fig.update_layout(
        width=width, height=height,
        font=_FONT,
        title_font=_FONT_TITL,
        margin=dict(l=72, r=36, t=64, b=72),
        paper_bgcolor="white",
        plot_bgcolor="white",
        legend=dict(
            bgcolor="rgba(255,255,255,0.9)",
            bordercolor="#cbd5e1",
            borderwidth=1,
            font=dict(size=11),
            title_font=dict(size=12),
        ),
    )
    fig.update_xaxes(**_AXIS_STYLE)
    fig.update_yaxes(**_AXIS_STYLE)
    return fig

def save_fig(fig, name, height=420, width=A4_W):
    """Style, export SVG + high-res PNG (scale=3), and show inline."""
    style_fig(fig, width=width, height=height)
    fig.write_image(str(EXPORTS_DIR / f"{name}.svg"))
    fig.write_image(str(EXPORTS_DIR / f"{name}.png"), scale=3)
    print(f"✓ exports/{name}.svg + .png")
    fig.show()

## Section 1 — Data Loading

In [ ]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def extract_tag_len(content, tag):
    """Return total character length of ALL <tag>\u2026</tag> blocks, else 0."""
    return sum(len(m.strip()) for m in re.findall(rf"<{tag}>(.*?)</{tag}>", content, re.DOTALL))

SPLITS = {
    "train":             DATA_DIR / "train_sft_v2.jsonl",         # v2 original
    "train_v3":   DATA_DIR / "train_sft_v3.jsonl",  # current (1,983 ex)
    "eval":              DATA_DIR / "eval_sft_v2.jsonl",
    "train_interleaved": DATA_DIR / "train_interleaved.jsonl",    # legacy
    "train_partB":       DATA_DIR / "train_partB.jsonl",
}

ALL_RECORDS = []   # full records — used by conversation renderer (Section 7)
rows = []

for split_name, path in SPLITS.items():
    if not path.exists():
        continue
    for rec in load_jsonl(path):
        meta = rec.get("metadata", {}).copy()
        msgs = rec.get("messages", [])
        asst = " ".join(m["content"] for m in msgs if m["role"] == "assistant")
        rows.append({
            **meta,
            "split":          split_name,
            "num_messages":   len(msgs),
            "response_chars": len(asst),
            "think_chars":    extract_tag_len(asst, "think"),
            "answer_chars":   extract_tag_len(asst, "answer"),
            "_idx":           len(ALL_RECORDS),
        })
        ALL_RECORDS.append(rec)

df = pd.DataFrame(rows)
df["category"]     = df["category"].fillna("(none)")
df["tool_profile"] = df["tool_profile"].fillna("(none)")
print(f"Loaded {len(df):,} records | {df['split'].nunique()} splits")
print(df.groupby("split").size().to_string())

## Section 2 — Dataset Overview

In [ ]:
# --- 2a: Summary metric cards ---
total      = len(df)
avg_score  = df["constitution_score"].dropna().mean() if "constitution_score" in df.columns else float("nan")
avg_len    = df["response_chars"].mean()
rev_pct    = df["revised"].dropna().mean() * 100 if "revised" in df.columns else float("nan")

metrics = [
    (f"{total:,}",            "Total Records"),
    (f"{df['split'].nunique()}", "Splits"),
    (f"{df['category'].nunique()}", "Categories"),
    (f"{avg_score:.3f}",      "Avg Score (train+eval)"),
    (f"{avg_len:,.0f}",       "Avg Response Length (chars)"),
    (f"{rev_pct:.0f}%",       "Revised (train+eval)"),
]
cards = "".join(f"""
  <div style="background:#f8fafc;border:1px solid #e2e8f0;border-radius:10px;
              padding:16px 24px;min-width:140px;text-align:center">
    <div style="font-size:26px;font-weight:700;color:#1e293b">{v}</div>
    <div style="font-size:12px;color:#64748b;margin-top:4px">{label}</div>
  </div>""" for v, label in metrics)
display(HTML(f"""
<div style="display:flex;gap:16px;flex-wrap:wrap;font-family:ui-sans-serif,sans-serif;
            margin:12px 0">{cards}</div>"""))

In [ ]:
# --- 2b: Category distribution donut ---
cat_counts = df["category"].value_counts().reset_index()
cat_counts.columns = ["category", "count"]
fig = px.pie(
    cat_counts, names="category", values="count",
    title="Training Data — Category Distribution",
    hole=0.42,
    color_discrete_sequence=PALETTE,
)
fig.update_traces(
    textposition="outside",
    textinfo="label+percent",
    textfont=dict(size=11),
    pull=[0.03] * len(cat_counts),
)
fig.update_layout(
    showlegend=True,
    legend=dict(title="Category", orientation="v", x=1.02, y=0.5),
)
save_fig(fig, "01_category_distribution", height=440)

In [ ]:
# --- 2c: Records per category per split (grouped bar) ---
split_cat = df.groupby(["split", "category"]).size().reset_index(name="count")
fig = px.bar(
    split_cat, x="category", y="count", color="split",
    barmode="group",
    title="Record Count per Category by Dataset Split",
    color_discrete_sequence=PALETTE,
    labels={"count": "Number of Records", "category": "Category", "split": "Split"},
)
fig.update_xaxes(tickangle=-35, title_text="Category")
fig.update_yaxes(title_text="Number of Records")
fig.update_layout(legend=dict(title="Split", x=1.01, y=1))
save_fig(fig, "02_split_category_counts", height=420)

## Section 3 — Constitution Quality Analysis

In [ ]:
# --- 3a: Constitution score distribution histogram ---
fig = px.histogram(
    df, x="constitution_score", nbins=20,
    title="Constitution Score Distribution across All Records",
    labels={"constitution_score": "Constitution Score (0–1)", "count": "Number of Records"},
    color_discrete_sequence=[PALETTE[0]],
)
fig.update_traces(marker_line_width=0.8, marker_line_color="white")
fig.update_xaxes(title_text="Constitution Score (0 = many violations, 1 = fully compliant)")
fig.update_yaxes(title_text="Number of Records")
fig.update_layout(bargap=0.06, showlegend=False)
save_fig(fig, "03_score_distribution", height=380)

In [ ]:
# --- 3b: Constitution score by category (box plot) ---
score_df = df[df["constitution_score"].notna()].copy()
fig = px.box(
    score_df, x="category", y="constitution_score",
    color="category", color_discrete_sequence=PALETTE,
    title="Constitution Score Distribution by Category",
    labels={"constitution_score": "Constitution Score (0–1)", "category": "Category"},
    points="outliers",
)
fig.update_xaxes(tickangle=-35, title_text="Category")
fig.update_yaxes(title_text="Constitution Score (0–1)", range=[0, 1.05])
fig.update_layout(showlegend=False)
save_fig(fig, "04_score_by_category", height=460)

In [ ]:
# --- 3c: Constitution violations in draft (histogram) ---
viol_df = df[df["constitution_violations_in_draft"].notna()].copy()
fig = px.histogram(
    viol_df, x="constitution_violations_in_draft",
    title="Number of Constitutional Violations in Initial Draft",
    labels={"constitution_violations_in_draft": "Violations in Draft", "count": "Number of Records"},
    color_discrete_sequence=[PALETTE[1]],
)
fig.update_traces(marker_line_width=0.8, marker_line_color="white")
fig.update_xaxes(title_text="Number of Violations in Draft Response", dtick=1)
fig.update_yaxes(title_text="Number of Records")
fig.update_layout(bargap=0.12, showlegend=False)
save_fig(fig, "05_violations_histogram", height=380)

In [ ]:
# --- 3d: Constitution score vs violations in draft (scatter) ---
sv_df = df[
    df["constitution_score"].notna() &
    df["constitution_violations_in_draft"].notna()
].copy()
fig = px.scatter(
    sv_df,
    x="constitution_violations_in_draft", y="constitution_score",
    color="category", color_discrete_sequence=PALETTE,
    title="Constitution Score vs. Violations in Draft",
    labels={
        "constitution_violations_in_draft": "Number of Violations in Draft",
        "constitution_score": "Final Constitution Score (0–1)",
        "category": "Category",
    },
    hover_data=["category", "tool_profile"],
    opacity=0.75,
)
fig.update_traces(marker=dict(size=7, line=dict(width=0.5, color="white")))
fig.update_xaxes(dtick=1, title_text="Number of Violations in Draft")
fig.update_yaxes(title_text="Final Constitution Score (0–1)", range=[0, 1.05])
fig.update_layout(legend=dict(title="Category", x=1.01, y=1))
save_fig(fig, "06_score_vs_violations", height=440)

## Section 4 — Tool Profile & Category Analysis

In [ ]:
# --- 4a: Tool profile distribution (donut) ---
tp_df = df[df["tool_profile"] != "(none)"].copy()
tp_counts = tp_df["tool_profile"].value_counts().reset_index()
tp_counts.columns = ["tool_profile", "count"]
fig = px.pie(
    tp_counts, names="tool_profile", values="count",
    hole=0.42,
    title="Tool Profile Distribution across Training Data",
    color_discrete_sequence=PALETTE,
)
fig.update_traces(
    textposition="outside",
    textinfo="label+percent",
    textfont=dict(size=11),
    pull=[0.03] * len(tp_counts),
)
fig.update_layout(
    showlegend=True,
    legend=dict(title="Tool Profile", orientation="v", x=1.02, y=0.5),
)
save_fig(fig, "07_tool_profile_distribution", height=420)

In [ ]:
# --- 4b: Category × tool_profile count heatmap ---
heat_df = df[(df["category"] != "(none)") & (df["tool_profile"] != "(none)")].copy()
pivot = (
    heat_df.groupby(["category", "tool_profile"]).size()
    .reset_index(name="count")
    .pivot(index="category", columns="tool_profile", values="count")
    .fillna(0).astype(int)
)
fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=pivot.columns.tolist(),
    y=pivot.index.tolist(),
    colorscale="Blues",
    text=pivot.values,
    texttemplate="%{text}",
    textfont=dict(size=11),
    showscale=True,
    colorbar=dict(title=dict(text="Count", side="right"), thickness=14, len=0.8),
))
fig.update_layout(
    title="Record Count by Category and Tool Profile",
    xaxis=dict(title="Tool Profile", tickfont=_FONT_TICK, title_font=_FONT_AXIS, side="bottom"),
    yaxis=dict(title="Category",     tickfont=_FONT_TICK, title_font=_FONT_AXIS),
    margin=dict(l=180, r=80, t=64, b=72),
)
save_fig(fig, "08_category_toolprofile_heatmap", height=480)

In [ ]:
# --- 4c: Average constitution score by tool profile ---
scored_df = df[(df["tool_profile"] != "(none)") & df["constitution_score"].notna()].copy()
avg_tp = scored_df.groupby("tool_profile")["constitution_score"].mean().reset_index()
avg_tp = avg_tp.sort_values("constitution_score")
fig = px.bar(
    avg_tp, x="tool_profile", y="constitution_score",
    title="Average Constitution Score by Tool Profile",
    color="tool_profile", color_discrete_sequence=PALETTE,
    labels={
        "constitution_score": "Average Constitution Score (0–1)",
        "tool_profile": "Tool Profile",
    },
    text_auto=".3f",
)
fig.update_traces(textposition="outside", textfont=dict(size=11))
fig.update_xaxes(title_text="Tool Profile")
fig.update_yaxes(title_text="Average Constitution Score (0–1)", range=[0, 1.1])
fig.update_layout(showlegend=False)
save_fig(fig, "09_avg_score_by_tool_profile", height=380)

## Section 5 — Response Quality Deep-Dive

In [ ]:
# --- 5a: Think-tag length vs answer-tag length (scatter) ---
has_think = df[(df["think_chars"] > 0) & (df["category"] != "(none)")].copy()
fig = px.scatter(
    has_think,
    x="think_chars", y="answer_chars",
    color="category", size="constitution_score", size_max=12,
    color_discrete_sequence=PALETTE,
    title="Reasoning Depth vs. Answer Length (bubble size = constitution score)",
    labels={
        "think_chars":        "<think> Block Length (chars)",
        "answer_chars":       "<answer> Block Length (chars)",
        "constitution_score": "Constitution Score",
        "category":           "Category",
    },
    hover_data=["tool_profile", "constitution_score", "constitution_violations_in_draft"],
    opacity=0.72,
)
fig.update_traces(marker=dict(line=dict(width=0.4, color="white")))
fig.update_xaxes(title_text="<think> Block Length (chars)")
fig.update_yaxes(title_text="<answer> Block Length (chars)")
fig.update_layout(legend=dict(title="Category", x=1.01, y=1))
save_fig(fig, "10_think_vs_answer_scatter", height=460)

In [ ]:
# --- 5b: Response length distribution by category (violin) ---
cat_df = df[df["category"] != "(none)"].copy()
fig = px.violin(
    cat_df, x="category", y="response_chars",
    color="category", color_discrete_sequence=PALETTE,
    box=True, points="outliers",
    title="Response Length Distribution by Category",
    labels={"response_chars": "Total Response Length (chars)", "category": "Category"},
)
fig.update_traces(meanline_visible=True)
fig.update_xaxes(tickangle=-35, title_text="Category")
fig.update_yaxes(title_text="Total Response Length (chars)")
fig.update_layout(showlegend=False)
save_fig(fig, "11_response_length_by_category", height=480)

In [ ]:
# --- 5c: Response length by tool profile (box) ---
tp_r_df = df[df["tool_profile"] != "(none)"].copy()
fig = px.box(
    tp_r_df, x="tool_profile", y="response_chars",
    color="tool_profile", color_discrete_sequence=PALETTE,
    points="outliers",
    title="Response Length by Tool Profile",
    labels={"response_chars": "Total Response Length (chars)", "tool_profile": "Tool Profile"},
)
fig.update_xaxes(title_text="Tool Profile")
fig.update_yaxes(title_text="Total Response Length (chars)")
fig.update_layout(showlegend=False)
save_fig(fig, "12_response_length_by_tool_profile", height=400)

In [ ]:
# --- 5d: Constitution score vs response length (OLS trendline) ---
ols_df = df[df["constitution_score"].notna() & (df["category"] != "(none)")].copy()
fig = px.scatter(
    ols_df, x="response_chars", y="constitution_score",
    color="category", color_discrete_sequence=PALETTE,
    trendline="ols",
    title="Constitution Score vs. Response Length (OLS trendline per category)",
    labels={
        "response_chars":     "Total Response Length (chars)",
        "constitution_score": "Constitution Score (0–1)",
        "category":           "Category",
    },
    hover_data=["tool_profile"],
    opacity=0.65,
)
fig.update_traces(
    selector=dict(mode="markers"),
    marker=dict(size=6, line=dict(width=0.4, color="white")),
)
fig.update_xaxes(title_text="Total Response Length (chars)")
fig.update_yaxes(title_text="Constitution Score (0–1)", range=[0, 1.05])
fig.update_layout(legend=dict(title="Category", x=1.01, y=1))
save_fig(fig, "13_score_vs_response_length", height=440)

## Section 6 — Cross-Model Performance Comparison

Compares base model vs custom (SFT-trained) model, with and without tools, on the same prompts.
Metrics extracted: number of turns, tool calls made, response length, reasoning depth (think-block length), and answer clarity (presence of `<answer>` tag).

In [ ]:
# --- 6a: Load reports and extract per-run metrics ---
def load_json_reports(pattern):
    return [json.load(open(p, encoding="utf-8"))
            for p in sorted(REPORTS_DIR.glob(pattern))]

benchmarks  = [r for r in load_json_reports("benchmark_*.json")
                if "conversation" in r or "runs" in r]  # exclude benchmark_compare_*.json
comparisons = load_json_reports("comparison_*.json")
print(f"Benchmarks: {len(benchmarks)} | Comparisons: {len(comparisons)}")

def run_metrics(conversation):
    """Extract scalar metrics from a single model run conversation."""
    asst_msgs  = [m for m in conversation if m["role"] == "assistant"]
    tool_calls = sum(
        1 for m in asst_msgs
        if "<tool>" in m.get("content", "") or "<tool_call>" in m.get("content", "")
    )
    has_answer = sum(1 for m in asst_msgs if "<answer>" in m.get("content", ""))
    avg_think  = (sum(extract_tag_len(m["content"], "think") for m in asst_msgs)
                  / len(asst_msgs) if asst_msgs else 0)
    avg_len    = (sum(len(m["content"]) for m in asst_msgs)
                  / len(asst_msgs) if asst_msgs else 0)
    return {
        "turns":      len(asst_msgs),
        "tool_calls": tool_calls,
        "has_answer": has_answer,
        "avg_think":  avg_think,
        "avg_len":    avg_len,
    }

In [ ]:
# --- 6b: Turn count — base vs custom across all comparison prompts ---
comp_rows = []
for c in comparisons:
    prompt = c.get("prompt", "")
    short  = (prompt[:50] + "…") if len(prompt) > 50 else prompt
    bm     = run_metrics(c.get("base_model_no_tools", {}).get("conversation", []))
    cm     = run_metrics(c.get("custom_model_output", {}).get("conversation", []))
    comp_rows.append({
        "prompt":            short,
        "Base (no tools)":   bm["turns"],
        "Custom (w/ tools)": cm["turns"],
    })
cdf = pd.DataFrame(comp_rows)

fig = go.Figure([
    go.Bar(name="Base Model (no tools)",   x=cdf["prompt"], y=cdf["Base (no tools)"],
           marker_color=PALETTE[0], text=cdf["Base (no tools)"],
           textposition="outside", textfont=dict(size=10)),
    go.Bar(name="Custom Model (w/ tools)", x=cdf["prompt"], y=cdf["Custom (w/ tools)"],
           marker_color=PALETTE[1], text=cdf["Custom (w/ tools)"],
           textposition="outside", textfont=dict(size=10)),
])
fig.update_layout(
    barmode="group",
    title="Response Turn Count: Base Model vs. Custom (SFT-trained) Model",
    legend=dict(title="Model Variant", x=1.01, y=1),
)
fig.update_xaxes(title_text="Evaluation Prompt", tickangle=-35)
fig.update_yaxes(title_text="Number of Conversation Turns")
save_fig(fig, "14_turn_count_comparison", height=440)

In [ ]:
# --- 6c: Multi-metric radar — aggregate comparison across all comparisons ---
def avg_metrics(key):
    vals = []
    for c in comparisons:
        conv = c.get(key, {}).get("conversation", [])
        if conv:
            vals.append(run_metrics(conv))
    if not vals:
        return {k: 0 for k in ["turns", "tool_calls", "has_answer", "avg_think", "avg_len"]}
    return pd.DataFrame(vals).mean().to_dict()

base_avg   = avg_metrics("base_model_no_tools")
custom_avg = avg_metrics("custom_model_output")

metrics_keys   = ["turns", "tool_calls", "has_answer", "avg_think", "avg_len"]
metrics_labels = ["Turns", "Tool Calls", "Answer Tags", "Avg Reasoning\n(chars)", "Avg Response\n(chars)"]

def normalise(vals):
    max_v = max(abs(v) for v in vals) or 1
    return [v / max_v for v in vals]

base_n   = normalise([base_avg[k]   for k in metrics_keys])
custom_n = normalise([custom_avg[k] for k in metrics_keys])

fig = go.Figure()
for label, vals, color in [
    ("Base Model (no tools)",   base_n,   PALETTE[0]),
    ("Custom Model (w/ tools)", custom_n, PALETTE[1]),
]:
    closed_vals   = vals   + [vals[0]]
    closed_labels = metrics_labels + [metrics_labels[0]]
    fig.add_trace(go.Scatterpolar(
        r=closed_vals, theta=closed_labels,
        fill="toself", name=label,
        line=dict(color=color, width=2),
        fillcolor=color.replace("rgb", "rgba").replace(")", ",0.15)") if color.startswith("rgb") else color,
    ))

fig.update_layout(
    title="Capability Radar: Base vs. Custom Model (Normalised Metrics)",
    polar=dict(
        radialaxis=dict(
            visible=True, range=[0, 1],
            tickfont=dict(size=9), tickvals=[0.25, 0.5, 0.75, 1.0],
            gridcolor="#e2e8f0",
        ),
        angularaxis=dict(tickfont=dict(size=11)),
    ),
    legend=dict(title="Model Variant", x=1.05, y=1, font=dict(size=11)),
    showlegend=True,
)
save_fig(fig, "15_model_radar", height=460)

In [ ]:
# --- 6d: Per-prompt metric table ---
table_rows = []
for c in comparisons:
    prompt = c.get("prompt", "")
    short  = (prompt[:60] + "…") if len(prompt) > 60 else prompt
    bm     = run_metrics(c.get("base_model_no_tools", {}).get("conversation", []))
    cm     = run_metrics(c.get("custom_model_output", {}).get("conversation", []))
    table_rows.append({
        "Prompt":             short,
        "Base Turns":         bm["turns"],
        "Custom Turns":       cm["turns"],
        "Custom Tool Calls":  cm["tool_calls"],
        "Custom Answer Tags": cm["has_answer"],
        "Base Avg Len":       f"{bm['avg_len']:.0f}",
        "Custom Avg Len":     f"{cm['avg_len']:.0f}",
    })
pd.DataFrame(table_rows).style.background_gradient(
    subset=["Base Turns", "Custom Turns", "Custom Tool Calls"], cmap="Blues"
)

In [ ]:
# --- 6e: Benchmark multi-run comparison ---
for bm in benchmarks:
    run_rows = []
    runs_dict = bm.get("runs", {})
    if runs_dict:
        # Old multi-run format: {"runs": {"label": {"conversation": [...]}}}
        for run_name, run_data in runs_dict.items():
            m = run_metrics(run_data.get("conversation", []))
            run_rows.append({"Run": run_name, **m})
    elif bm.get("conversation"):
        # New flat format: {"conversation": [...], "summary": {}}
        label = bm.get("model_label", bm.get("server_url", "model"))
        m = run_metrics(bm["conversation"])
        run_rows.append({"Run": label, **m})
    if not run_rows:
        continue
    bdf    = pd.DataFrame(run_rows)
    colors = PALETTE[:len(bdf)]
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=("Conversation Turns", "Tool Calls Made", "Avg Response Length (chars)"),
    )
    for col, (y_col, y_title) in enumerate(
        [("turns", "Turns"), ("tool_calls", "Tool Calls"), ("avg_len", "Avg Length (chars)")],
        start=1,
    ):
        fig.add_trace(go.Bar(
            x=bdf["Run"], y=bdf[y_col],
            marker_color=colors,
            text=bdf[y_col].round(1),
            textposition="outside",
            textfont=dict(size=10),
            showlegend=False,
        ), row=1, col=col)
        fig.update_yaxes(title_text=y_title, row=1, col=col)
        fig.update_xaxes(title_text="Run",   row=1, col=col, tickangle=-20)

    ts = bm.get("timestamp", "")
    fig.update_layout(title=f"Benchmark Multi-Run Comparison — {ts}")
    save_fig(fig, f"16_benchmark_{ts}", height=420)

## Section 7 — Conversation Viewer

Light-mode, dissertation-ready conversation renderer with syntax highlighting for `<think>`, `<answer>`, and `<tool>` tags. Use the interactive browser to explore training samples by category and tool profile.

In [ ]:
# --- 7a: Conversation renderer ---
_ROLE_STYLE = {
    "system":    ("System",    "#eff6ff", "#1d4ed8"),
    "user":      ("User",      "#f0fdf4", "#16a34a"),
    "assistant": ("Assistant", "#faf5ff", "#7c3aed"),
}

def _highlight_tags(text):
    text = re.sub(
        r"<think>(.*?)</think>",
        lambda m: (
            "<details open style='margin:4px 0'>"
            "<summary style='color:#6366f1;font-weight:600;cursor:pointer'>&#x1f4ad; Reasoning</summary>"
            "<pre style='white-space:pre-wrap;background:#f8f7ff;padding:10px;border-radius:6px;"
            "font-size:13px;color:#3730a3;margin:4px 0'>"
            + m.group(1) + "</pre></details>"
        ),
        text, flags=re.DOTALL
    )
    text = re.sub(
        r"<answer>(.*?)</answer>",
        lambda m: (
            "<div style='background:#f0fdf4;border-left:3px solid #22c55e;"
            "padding:8px 12px;margin:6px 0;border-radius:0 6px 6px 0'>"
            "<span style='font-weight:600;color:#16a34a'>Answer: </span>"
            + m.group(1) + "</div>"
        ),
        text, flags=re.DOTALL
    )
    text = re.sub(
        r"<tool>(.*?)</tool>",
        lambda m: (
            "<code style='background:#fff7ed;border:1px solid #fed7aa;"
            "padding:3px 8px;border-radius:4px;font-size:13px'>"
            "&#x1f527; " + m.group(1) + "</code>"
        ),
        text, flags=re.DOTALL
    )
    text = re.sub(
        r"<tool_call>(.*?)</tool_call>",
        lambda m: (
            "<code style='background:#fff7ed;border:1px solid #fed7aa;"
            "padding:3px 8px;border-radius:4px;font-size:13px'>"
            "&#x1f527; [native] " + m.group(1).strip()[:120] + "</code>"
        ),
        text, flags=re.DOTALL
    )
    return text

def render_conversation(messages, title="Conversation", score=None, category=None):
    meta = ""
    if category or score is not None:
        parts = []
        if category:          parts.append(f"<strong>Category:</strong> {category}")
        if score is not None: parts.append(f"<strong>Score:</strong> {score:.3f}")
        meta = f"<p style='font-size:12px;color:#64748b;margin:4px 0 10px'>{' · '.join(parts)}</p>"

    bubbles = ""
    for msg in messages:
        role           = msg.get("role", "assistant")
        label, bg, acc = _ROLE_STYLE.get(role, ("?", "#fff", "#000"))
        content        = _highlight_tags(msg.get("content", ""))
        bubbles += (
            f"<div style='background:{bg};border:1px solid #e2e8f0;border-radius:8px;"
            f"padding:10px 14px'>"
            f"<div style='font-size:11px;font-weight:700;color:{acc};"
            f"text-transform:uppercase;letter-spacing:.06em;margin-bottom:6px'>{label}</div>"
            f"<div style='font-size:14px;color:#1e293b;line-height:1.6'>{content}</div>"
            f"</div>"
        )

    html = (
        f"<div style='font-family:ui-sans-serif,system-ui,sans-serif;max-width:860px;"
        f"border:1px solid #e2e8f0;border-radius:12px;overflow:hidden;"
        f"box-shadow:0 1px 4px rgba(0,0,0,0.06)'>"
        f"<div style='background:#f8fafc;padding:12px 18px;border-bottom:1px solid #e2e8f0'>"
        f"<h3 style='margin:0;font-size:16px;color:#1e293b'>{title}</h3>{meta}</div>"
        f"<div style='padding:14px 16px;display:flex;flex-direction:column;gap:10px'>"
        f"{bubbles}</div></div>"
    )
    return HTML(html)

In [ ]:
# --- 7b: Interactive training sample browser ---
from ipywidgets import interact, Dropdown, IntSlider

_CATEGORIES    = sorted(df[df["category"] != "(none)"]["category"].unique().tolist())
_TOOL_PROFILES = ["(any)"] + sorted(df[df["tool_profile"] != "(none)"]["tool_profile"].unique().tolist())

def browse_samples(category=_CATEGORIES[0], tool_profile="(any)", seed=42):
    subset = df[df["category"] == category]
    if tool_profile != "(any)":
        subset = subset[subset["tool_profile"] == tool_profile]
    if subset.empty:
        display(HTML("<p style='color:#ef4444;font-family:sans-serif'>No matching records.</p>"))
        return
    row = subset.sample(1, random_state=seed).iloc[0]
    rec = ALL_RECORDS[row["_idx"]]
    display(render_conversation(
        rec["messages"],
        title=f"{row['category']} · {row['tool_profile']}",
        score=row.get("constitution_score"),
        category=row["category"],
    ))

interact(
    browse_samples,
    category=Dropdown(options=_CATEGORIES, description="Category:"),
    tool_profile=Dropdown(options=_TOOL_PROFILES, description="Tool Profile:"),
    seed=IntSlider(min=0, max=100, value=42, description="Seed:"),
)

In [ ]:
# --- 7c: Side-by-side comparison viewer ---
from ipywidgets import interact, Dropdown

_COMP_LABELS = [
    ((c.get("prompt", "")[:60] + "…") if len(c.get("prompt", "")) > 60 else c.get("prompt", ""))
    for c in comparisons
]

def show_comparison(prompt_label=_COMP_LABELS[0] if _COMP_LABELS else ""):
    idx = _COMP_LABELS.index(prompt_label) if prompt_label in _COMP_LABELS else 0
    c   = comparisons[idx]
    display(HTML(
        f"<h3 style='font-family:sans-serif;margin:12px 0 4px'>"
        f"Prompt: <em style='font-weight:400'>{c.get('prompt', '')}</em></h3>"
    ))
    display(render_conversation(
        c["base_model_no_tools"]["conversation"], "Base Model (No Tools)"))
    display(HTML("<br>"))
    display(render_conversation(
        c["custom_model_output"]["conversation"], "Custom Model (With Tools)"))

if _COMP_LABELS:
    interact(show_comparison, prompt_label=Dropdown(options=_COMP_LABELS, description="Prompt:"))
else:
    display(HTML("<p style='color:#64748b'>No comparison reports found in reports/</p>"))

## Section 8 — ROUGE Scores

Compares ROUGE-1, ROUGE-2, ROUGE-L F1 across checkpoints for two reference sources: eval split gold responses and constitution probe baseline. Load `reports/rouge_*.json` produced by `2_model_trainer.py publish()`.

In [ ]:
# --- 8a: Load ROUGE reports ---
rouge_reports = []
for p in sorted(REPORTS_DIR.glob("rouge_*.json")):
    with open(p, encoding="utf-8") as f:
        rouge_reports.append(json.load(f))

print(f"ROUGE reports: {len(rouge_reports)}")
for r in rouge_reports:
    ev  = "✓" if r.get("eval_split_rouge")     else "✗"
    pb  = "✓" if r.get("probe_baseline_rouge") else "✗"
    rwd = f"{r['grpo_held_out_reward']:.4f}" if r.get("grpo_held_out_reward") is not None else "—"
    print(f"  {r['checkpoint']:35}  eval={ev}  probe={pb}  grpo_reward={rwd}")

In [ ]:
# --- 8b: ROUGE F1 — eval split gold responses ---
def _rouge_to_df(reports, key):
    rows = []
    for r in reports:
        block = r.get(key)
        if not block:
            continue
        rows.append({
            "Checkpoint": r["checkpoint"],
            "ROUGE-1":    block["rouge1"]["fmeasure"],
            "ROUGE-2":    block["rouge2"]["fmeasure"],
            "ROUGE-L":    block["rougeL"]["fmeasure"],
        })
    return pd.DataFrame(rows)

eval_df = _rouge_to_df(rouge_reports, "eval_split_rouge")

if not eval_df.empty:
    melted = eval_df.melt(id_vars="Checkpoint", var_name="Metric", value_name="F1 Score")
    fig = px.bar(
        melted, x="Checkpoint", y="F1 Score", color="Metric",
        barmode="group",
        title="ROUGE F1 Score — Eval Split (model output vs. gold responses)",
        color_discrete_sequence=PALETTE,
        text_auto=".3f",
        labels={"Checkpoint": "Model Checkpoint", "F1 Score": "F1 Score (0–1)"},
    )
    fig.update_traces(textposition="outside", textfont=dict(size=10))
    fig.update_xaxes(title_text="Model Checkpoint", tickangle=-20)
    fig.update_yaxes(title_text="F1 Score (0–1)", range=[0, 1.1])
    fig.update_layout(legend=dict(title="ROUGE Metric", x=1.01, y=1))
    save_fig(fig, "17_rouge_eval_split", height=420)
else:
    print("No eval-split ROUGE data yet — run 2_model_trainer.py to generate reports/rouge_*.json")

In [ ]:
# --- 8c: ROUGE F1 — probe baseline (constitution drift indicator) ---
probe_df = _rouge_to_df(rouge_reports, "probe_baseline_rouge")

if not probe_df.empty:
    melted = probe_df.melt(id_vars="Checkpoint", var_name="Metric", value_name="F1 Score")
    fig = px.bar(
        melted, x="Checkpoint", y="F1 Score", color="Metric",
        barmode="group",
        title="ROUGE F1 Score — Constitution Probe Baseline (constitutional drift indicator)",
        color_discrete_sequence=PALETTE,
        text_auto=".3f",
        labels={"Checkpoint": "Model Checkpoint", "F1 Score": "F1 Score (0–1)"},
    )
    fig.update_traces(textposition="outside", textfont=dict(size=10))
    fig.update_xaxes(title_text="Model Checkpoint", tickangle=-20)
    fig.update_yaxes(title_text="F1 Score (0–1)", range=[0, 1.1])
    fig.update_layout(legend=dict(title="ROUGE Metric", x=1.01, y=1))
    save_fig(fig, "18_rouge_probe_baseline", height=420)
else:
    print("No probe-baseline ROUGE data yet.")

In [ ]:
# --- 8d: GRPO held-out reward score by checkpoint ---
reward_rows = [
    {"Checkpoint": r["checkpoint"], "Held-out Reward": r["grpo_held_out_reward"]}
    for r in rouge_reports
    if r.get("grpo_held_out_reward") is not None
]
if reward_rows:
    rdf = pd.DataFrame(reward_rows)
    fig = px.bar(
        rdf, x="Checkpoint", y="Held-out Reward",
        title="GRPO Held-Out Reward Score (10 % held-out prompts, full composite reward function)",
        color_discrete_sequence=[PALETTE[2]],
        text_auto=".4f",
        labels={"Checkpoint": "Model Checkpoint", "Held-out Reward": "Mean Reward (0–1)"},
    )
    fig.update_traces(textposition="outside", textfont=dict(size=11))
    fig.update_xaxes(title_text="Model Checkpoint", tickangle=-20)
    fig.update_yaxes(title_text="Mean Reward (0–1)", range=[0, 1.1])
    fig.update_layout(showlegend=False)
    save_fig(fig, "19_grpo_held_out_reward", height=380)
else:
    print("No GRPO held-out reward data yet — run GRPO training to generate.")

## Section 9 — Training Loss Curves

SFT loss curves loaded from `models/checkpoint_sft*/loss_history.json`. GRPO reward/loss curves loaded from `models/checkpoint_grpo*/grpo_loss_history.json`.

In [ ]:
# --- 9a: Discover and load loss/reward history files ---
models_dir = Path("models")

sft_histories  = {}
grpo_histories = {}

for p in sorted(models_dir.glob("checkpoint_sft*/loss_history.json")):
    with open(p, encoding="utf-8") as f:
        sft_histories[p.parent.name] = json.load(f)

for p in sorted(models_dir.glob("checkpoint_grpo*/grpo_loss_history.json")):
    with open(p, encoding="utf-8") as f:
        grpo_histories[p.parent.name] = json.load(f)

print(f"SFT checkpoints:  {list(sft_histories.keys())  or ['none found']}")
print(f"GRPO checkpoints: {list(grpo_histories.keys()) or ['none found']}")

In [ ]:
# --- 9b: SFT train + eval loss curves ---
for label, history in sft_histories.items():
    train_rows = [h for h in history if "loss" in h and "eval_loss" not in h]
    eval_rows  = [h for h in history if "eval_loss" in h]

    if not train_rows:
        print(f"  {label}: no training-loss entries found")
        continue

    fig = make_subplots(rows=1, cols=2, subplot_titles=("Loss Curves", "Eval Loss (zoomed)"))

    fig.add_trace(go.Scatter(
        x=[h["step"] for h in train_rows],
        y=[h["loss"]  for h in train_rows],
        mode="lines", name="Train Loss",
        line=dict(color=PALETTE[0], width=2),
    ), row=1, col=1)

    if eval_rows:
        ev_x = [h["step"]      for h in eval_rows]
        ev_y = [h["eval_loss"] for h in eval_rows]
        fig.add_trace(go.Scatter(
            x=ev_x, y=ev_y,
            mode="lines+markers", name="Eval Loss",
            line=dict(color=PALETTE[1], width=2),
            marker=dict(size=6),
        ), row=1, col=1)
        fig.add_trace(go.Scatter(
            x=ev_x, y=ev_y,
            mode="lines+markers", name="Eval Loss",
            line=dict(color=PALETTE[1], width=2),
            marker=dict(size=6),
            showlegend=False,
        ), row=1, col=2)

    fig.update_yaxes(title_text="Cross-Entropy Loss", row=1, col=1)
    fig.update_yaxes(title_text="Eval Loss",          row=1, col=2)
    fig.update_xaxes(title_text="Training Step",      row=1, col=1)
    fig.update_xaxes(title_text="Training Step",      row=1, col=2)
    fig.update_layout(
        title=f"SFT Training Loss Curves — {label}",
        legend=dict(title="Split", x=1.01, y=1),
    )
    safe = label.replace("/", "_")
    save_fig(fig, f"20_sft_loss_{safe}", height=420)

In [ ]:
# --- 9d: SFT grad norm + learning rate (training stability) ---
import statistics as _stats

for label, history in sft_histories.items():
    train_rows = [h for h in history if "loss" in h and "eval_loss" not in h]
    grad_rows  = [h for h in train_rows if "grad_norm" in h]
    lr_rows    = [h for h in train_rows if "learning_rate" in h]

    if not grad_rows and not lr_rows:
        print(f"  {label}: no grad_norm / learning_rate in history")
        continue

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            "Gradient Norm  (spikes = instability)",
            "Learning Rate Schedule  (warmup → decay)",
        ),
    )

    if grad_rows:
        steps = [h["step"]      for h in grad_rows]
        norms = [h["grad_norm"] for h in grad_rows]
        fig.add_trace(go.Scatter(
            x=steps, y=norms, mode="lines",
            name="Grad Norm",
            line=dict(color=PALETTE[4], width=1.8),
        ), row=1, col=1)

        median_norm = _stats.median(norms)
        spike_steps = [s for s, n in zip(steps, norms) if n > 3 * median_norm]
        spike_norms = [n for s, n in zip(steps, norms) if n > 3 * median_norm]
        if spike_steps:
            fig.add_trace(go.Scatter(
                x=spike_steps, y=spike_norms, mode="markers",
                name=f"Spike (> 3× median = {median_norm:.2f})",
                marker=dict(color="#ef4444", size=9, symbol="x", line=dict(width=2)),
            ), row=1, col=1)
            print(f"  {label}: {len(spike_steps)} spike(s) at steps "
                  f"{spike_steps[:5]}{'…' if len(spike_steps) > 5 else ''}")

    if lr_rows:
        fig.add_trace(go.Scatter(
            x=[h["step"]          for h in lr_rows],
            y=[h["learning_rate"] for h in lr_rows],
            mode="lines", name="Learning Rate",
            line=dict(color=PALETTE[5], width=1.8),
            showlegend=False,
        ), row=1, col=2)

    fig.update_yaxes(title_text="Gradient Norm",     row=1, col=1)
    fig.update_yaxes(title_text="Learning Rate",     row=1, col=2, tickformat=".2e")
    fig.update_xaxes(title_text="Training Step",     row=1, col=1)
    fig.update_xaxes(title_text="Training Step",     row=1, col=2)
    fig.update_layout(
        title=f"SFT Training Stability — {label}",
        legend=dict(title="", x=0.02, y=0.98, bgcolor="rgba(255,255,255,0.85)"),
    )
    safe = label.replace("/", "_")
    save_fig(fig, f"24_sft_stability_{safe}", height=400)

In [ ]:
# --- 9e: GRPO grad norm + KL divergence from SFT reference + learning rate ---
for label, history in grpo_histories.items():
    train_rows = [h for h in history if "loss" in h and "eval_loss" not in h]
    grad_rows  = [h for h in train_rows if "grad_norm" in h]
    kl_rows    = [h for h in history if "kl" in h or "objective/kl" in h]
    lr_rows    = [h for h in train_rows if "learning_rate" in h]

    if not grad_rows and not kl_rows:
        print(f"  {label}: no grad_norm / kl entries in history")
        continue

    kl_key = "objective/kl" if (kl_rows and "objective/kl" in kl_rows[0]) else "kl"

    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=(
            "Gradient Norm",
            "KL from SFT Reference  (constitutional drift proxy)",
            "Learning Rate Schedule",
        ),
    )

    if grad_rows:
        fig.add_trace(go.Scatter(
            x=[h["step"]      for h in grad_rows],
            y=[h["grad_norm"] for h in grad_rows],
            mode="lines", name="Grad Norm",
            line=dict(color=PALETTE[4], width=1.8),
            showlegend=False,
        ), row=1, col=1)

    if kl_rows:
        kl_vals = [h[kl_key] for h in kl_rows]
        fig.add_trace(go.Scatter(
            x=[h["step"] for h in kl_rows],
            y=kl_vals,
            mode="lines", name="KL Divergence",
            line=dict(color=PALETTE[3], width=1.8),
            showlegend=False,
        ), row=1, col=2)
        final_kl = kl_vals[-1]
        status   = "within bounds" if final_kl < 0.1 else "HIGH — possible drift"
        print(f"  {label} — final KL = {final_kl:.4f}  ({status})")

    if lr_rows:
        fig.add_trace(go.Scatter(
            x=[h["step"]          for h in lr_rows],
            y=[h["learning_rate"] for h in lr_rows],
            mode="lines", name="Learning Rate",
            line=dict(color=PALETTE[5], width=1.8),
            showlegend=False,
        ), row=1, col=3)

    fig.update_yaxes(title_text="Gradient Norm",  row=1, col=1)
    fig.update_yaxes(title_text="KL Divergence",  row=1, col=2)
    fig.update_yaxes(title_text="Learning Rate",  row=1, col=3, tickformat=".2e")
    for c in range(1, 4):
        fig.update_xaxes(title_text="Training Step", row=1, col=c)

    fig.update_layout(title=f"GRPO Training Stability — {label}")
    safe = label.replace("/", "_")
    save_fig(fig, f"25_grpo_stability_{safe}", height=400)

In [ ]:
# --- 9c: GRPO per-component reward breakdown + policy loss ---
COMPONENT_META = {
    "format_reward":          ("Format",          PALETTE[0]),
    "accuracy_reward":        ("Accuracy",        PALETTE[1]),
    "tool_integrity_reward":  ("Tool Integrity",  PALETTE[2]),
    "constitution_reward":    ("Constitution",    PALETTE[3]),
    "format_reward_c":        ("Format (Abl. C)", PALETTE[0]),
    "accuracy_reward_c":      ("Accuracy (Abl. C)", PALETTE[1]),
}

for label, history in grpo_histories.items():
    loss_rows  = [h for h in history if "loss" in h and "eval_loss" not in h]
    total_rows = [h for h in history if "rewards/mean" in h]
    present    = {
        k: meta for k, meta in COMPONENT_META.items()
        if any(f"rewards/{k}_mean" in h for h in history)
    }

    if not loss_rows and not total_rows:
        print(f"  {label}: no usable entries in grpo_loss_history.json")
        continue

    n_panels   = 2 + len(present)
    subtitles  = (
        ["Policy Loss", "Total Reward"]
        + [f"{meta[0]} Reward" for meta in present.values()]
    )
    fig = make_subplots(rows=1, cols=n_panels, subplot_titles=subtitles)

    if loss_rows:
        fig.add_trace(go.Scatter(
            x=[h["step"] for h in loss_rows],
            y=[h["loss"]  for h in loss_rows],
            mode="lines", name="Policy Loss",
            line=dict(color="#64748b", width=2),
            showlegend=False,
        ), row=1, col=1)

    if total_rows:
        fig.add_trace(go.Scatter(
            x=[h["step"]         for h in total_rows],
            y=[h["rewards/mean"] for h in total_rows],
            mode="lines", name="Total Reward",
            line=dict(color="#0f172a", width=2),
            showlegend=False,
        ), row=1, col=2)

    for col_off, (key, (comp_label, color)) in enumerate(present.items(), start=3):
        comp_rows = [h for h in history if f"rewards/{key}_mean" in h]
        fig.add_trace(go.Scatter(
            x=[h["step"]                 for h in comp_rows],
            y=[h[f"rewards/{key}_mean"]  for h in comp_rows],
            mode="lines", name=comp_label,
            line=dict(color=color, width=2),
            showlegend=False,
        ), row=1, col=col_off)

    fig.update_yaxes(title_text="Loss",   row=1, col=1)
    fig.update_yaxes(title_text="Reward", row=1, col=2)
    for c in range(3, n_panels + 1):
        fig.update_yaxes(title_text="Reward (weighted)", row=1, col=c)
    for c in range(1, n_panels + 1):
        fig.update_xaxes(title_text="Step", row=1, col=c)

    panel_w = 240
    fig.update_layout(title=f"GRPO Training Curves — {label}")
    safe = label.replace("/", "_")
    save_fig(fig, f"21_grpo_curves_{safe}", height=400, width=max(A4_W, n_panels * panel_w))

    # Print final component values
    last = {k: v for h in history for k, v in h.items()}
    parts = [
        f"{COMPONENT_META[k][0]}={last[f'rewards/{k}_mean']:.4f}"
        for k in present if f"rewards/{k}_mean" in last
    ]
    if parts:
        print(f"  {label} — total={last.get('rewards/mean', '—')}  |  " + "  ".join(parts))

## Section 10 — Safety Metrics

Three checks that are central to the dissertation's trustworthiness argument: perplexity (human-readable loss), overfitting gap (train vs eval), and format compliance rate on the gold eval split.

In [ ]:
import math

# --- 10a: Perplexity + overfitting gap (eval_loss − train_loss) ---
for label, history in sft_histories.items():
    train_rows = [h for h in history if "loss" in h and "eval_loss" not in h]
    eval_rows  = [h for h in history if "eval_loss" in h]

    if not train_rows:
        print(f"{label}: no training loss entries")
        continue

    train_steps = [h["step"] for h in train_rows]
    train_loss  = [h["loss"] for h in train_rows]
    eval_steps  = [h["step"]      for h in eval_rows]
    eval_loss   = [h["eval_loss"] for h in eval_rows]
    perplexity  = [math.exp(min(l, 20)) for l in eval_loss]

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            "Train vs. Eval Loss  (widening gap = overfitting)",
            "Eval Perplexity  (lower is better)",
        ),
    )

    fig.add_trace(go.Scatter(
        x=train_steps, y=train_loss,
        mode="lines", name="Train Loss",
        line=dict(color=PALETTE[0], width=2),
    ), row=1, col=1)

    if eval_rows:
        fig.add_trace(go.Scatter(
            x=eval_steps, y=eval_loss,
            mode="lines+markers", name="Eval Loss",
            line=dict(color=PALETTE[1], width=2),
            marker=dict(size=6),
        ), row=1, col=1)
        fig.add_trace(go.Scatter(
            x=eval_steps, y=perplexity,
            mode="lines+markers", name="Perplexity",
            line=dict(color=PALETTE[2], width=2),
            marker=dict(size=6),
            showlegend=False,
        ), row=1, col=2)

    fig.update_yaxes(title_text="Cross-Entropy Loss",           row=1, col=1)
    fig.update_yaxes(title_text="Perplexity  exp(eval_loss)",   row=1, col=2)
    fig.update_xaxes(title_text="Training Step",                row=1, col=1)
    fig.update_xaxes(title_text="Training Step",                row=1, col=2)
    fig.update_layout(
        title=f"SFT Safety: Loss Gap & Perplexity — {label}",
        legend=dict(title="Split", x=1.01, y=1),
    )
    safe = label.replace("/", "_")
    save_fig(fig, f"22_sft_perplexity_{safe}", height=420)

    if eval_rows and train_rows:
        gap = eval_loss[-1] - train_loss[-1]
        print(f"  {label} — train_loss={train_loss[-1]:.4f}  "
              f"eval_loss={eval_loss[-1]:.4f}  gap={gap:+.4f}  "
              f"perplexity={perplexity[-1]:.2f}")

In [ ]:
# --- 10b: Format compliance rate on gold eval split ---
def check_format_compliance(records):
    results = []
    for rec in records:
        msgs = rec.get("messages", [])
        asst = " ".join(m["content"] for m in msgs if m["role"] == "assistant")
        meta = rec.get("metadata", {})
        has_think = bool(re.search(r"<think>",  asst, re.IGNORECASE))
        has_cap   = "CAPABILITY_CHECK" in asst
        has_ans   = bool(re.search(r"<answer>", asst, re.IGNORECASE))
        results.append({
            "category":     meta.get("category") or meta.get("question_type", "unknown"),
            "tool_profile": meta.get("tool_profile", "unknown"),
            "has_think":    has_think,
            "has_cap":      has_cap,
            "has_answer":   has_ans,
            "compliant":    has_think and has_cap and has_ans,
        })
    return pd.DataFrame(results)

eval_path = DATA_DIR / "eval_sft_v2.jsonl"
fallback  = DATA_DIR / "train_sft_v3.jsonl"
src_path  = eval_path if eval_path.exists() else (fallback if fallback.exists() else None)
src_label = "Eval Split (eval_sft_v2)" if eval_path.exists() else "Train v3 Robust (fallback)"

if src_path:
    compliance_df = check_format_compliance(load_jsonl(src_path))
    overall = compliance_df["compliant"].mean() * 100
    print(f"Format compliance — {src_label}")
    print(f"  Overall:    {overall:.1f}%  ({compliance_df['compliant'].sum()}/{len(compliance_df)} records)")
    print(f"  <think>:    {compliance_df['has_think'].mean()*100:.1f}%")
    print(f"  CAPABILITY: {compliance_df['has_cap'].mean()*100:.1f}%")
    print(f"  <answer>:   {compliance_df['has_answer'].mean()*100:.1f}%")

    cat_comp = (
        compliance_df.groupby("category")["compliant"]
        .agg(["sum", "count"])
        .assign(rate=lambda x: x["sum"] / x["count"] * 100)
        .reset_index()
        .rename(columns={"sum": "Compliant", "count": "Total", "rate": "Compliance (%)"})
        .sort_values("Compliance (%)")
    )

    fig = px.bar(
        cat_comp, x="category", y="Compliance (%)",
        title=f"Format Compliance Rate by Category — {src_label} (overall {overall:.1f}%)",
        color="Compliance (%)",
        color_continuous_scale=[[0, "#ef4444"], [0.8, "#f59e0b"], [1, "#22c55e"]],
        range_color=[80, 100],
        text_auto=".1f",
        labels={"category": "Category", "Compliance (%)": "Compliance (%)"},
        custom_data=["Compliant", "Total"],
    )
    fig.update_traces(
        textposition="outside",
        textfont=dict(size=11),
        hovertemplate="<b>%{x}</b><br>Compliant: %{customdata[0]}/%{customdata[1]}<br>Rate: %{y:.1f}%<extra></extra>",
    )
    fig.update_xaxes(tickangle=-35, title_text="Category")
    fig.update_yaxes(title_text="Format Compliance Rate (%)", range=[0, 115])
    fig.update_layout(
        coloraxis_showscale=True,
        coloraxis_colorbar=dict(title="Compliance (%)", thickness=14),
        showlegend=False,
    )
    save_fig(fig, "23_format_compliance_by_category", height=420)
else:
    print("No eval or train_interleaved dataset found — run sft_dataset_assembler.py first")